# 23 — Surviving Residues

## What survives pruning?

Notebook 17 reframed pruning as revision rather than reset.

Notebook 23 asks:

> Which structures or capabilities are most likely to survive compression and retraining?

Outputs:

```text
results/23_surviving_residues.csv
figures/23_surviving_residues.png
```

No model downloads. No GPU. No pruning jobs.

In [ ]:
# Cell 1 — locate or clone repo

from pathlib import Path
import os
import sys
import subprocess

REPO_URL = "https://github.com/thinkthoughts/pruning-rml.git"
REPO_NAME = "pruning-rml"

cwd = Path.cwd().resolve()

if cwd.name == REPO_NAME:
    repo = cwd
elif cwd.name == "notebooks" and cwd.parent.name == REPO_NAME:
    repo = cwd.parent
elif Path("/content/pruning-rml").exists():
    repo = Path("/content/pruning-rml")
elif (cwd / REPO_NAME).exists():
    repo = cwd / REPO_NAME
else:
    target = Path("/content") if Path("/content").exists() else cwd
    os.chdir(target)
    subprocess.run(["git", "clone", REPO_URL], check=True)
    repo = target / REPO_NAME

os.chdir(repo)

src = repo / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

print("repo:", repo)
print("src :", src)
print("src exists:", src.exists())

In [ ]:
# Cell 2 — ensure helper files

pkg = repo / "src" / "pruning_rml"
pkg.mkdir(parents=True, exist_ok=True)

(pkg / "__init__.py").write_text(
    '__version__ = "0.1.0"\n',
    encoding="utf-8",
)

(pkg / "paths.py").write_text(
    '''from pathlib import Path

def ensure_dirs(root, names=("results", "figures", "reports")):
    root = Path(root)
    outputs = {}
    for name in names:
        path = root / name
        path.mkdir(parents=True, exist_ok=True)
        outputs[name] = path
    return outputs
''',
    encoding="utf-8",
)

(pkg / "residues.py").write_text(
    '''def residue_strength(value):
    return value

def survives(value, threshold=0.5):
    return value >= threshold

def retention_label(value):
    if value >= 0.8:
        return "high"
    if value >= 0.5:
        return "medium"
    return "low"
''',
    encoding="utf-8",
)

for module_name in [
    "pruning_rml",
    "pruning_rml.paths",
    "pruning_rml.residues",
]:
    if module_name in sys.modules:
        del sys.modules[module_name]

print("src helpers ready")
print("package files:", sorted(p.name for p in pkg.iterdir()))

In [ ]:
# Cell 3 — imports and output folders

import pandas as pd
import matplotlib.pyplot as plt

from pruning_rml.paths import ensure_dirs
from pruning_rml.residues import retention_label, survives

outputs = ensure_dirs(repo)
results = outputs["results"]
figures = outputs["figures"]

print("results:", results)
print("figures:", figures)

## Residue schema

In this repo, a residue is a structure or capability that remains useful after revision.

For pruning, the practical question is:

```text
larger pretrained model
    ↓
prune
    ↓
retrain
    ↓
evaluate what survived
```

This notebook uses toy values to define the vocabulary before real pruning experiments.

In [ ]:
# Cell 4 — build surviving residues table

rows = [
    {
        "residue": "attention structure",
        "description": "patterns that route information across tokens",
        "retention_score": 0.92,
        "risk": "coarse layer removal may disrupt routing",
    },
    {
        "residue": "token patterns",
        "description": "frequent learned token relationships",
        "retention_score": 0.88,
        "risk": "aggressive pruning may reduce fluency",
    },
    {
        "residue": "feature circuits",
        "description": "distributed features supporting tasks or behaviors",
        "retention_score": 0.74,
        "risk": "structured pruning may remove useful subcircuits",
    },
    {
        "residue": "task heuristics",
        "description": "learned shortcuts that support benchmark performance",
        "retention_score": 0.63,
        "risk": "retraining may shift heuristic behavior",
    },
    {
        "residue": "memorized details",
        "description": "specific facts or examples retained from pretraining",
        "retention_score": 0.38,
        "risk": "likely fragile under compression",
    },
    {
        "residue": "unused parameters",
        "description": "low-value structure that may be removable",
        "retention_score": 0.18,
        "risk": "removal may be beneficial if correctly identified",
    },
]

df = pd.DataFrame(rows)
df["retention_label"] = df["retention_score"].apply(retention_label)
df["survives_threshold"] = df["retention_score"].apply(survives)

df

In [ ]:
# Cell 5 — save CSV

csv_path = results / "23_surviving_residues.csv"
df.to_csv(csv_path, index=False)

print("saved:", csv_path)
print("exists:", csv_path.exists())

## Visual summary

The figure ranks candidate residues by expected retention.

Working hypothesis:

> pruning succeeds when useful residues survive compression strongly enough that retraining can refine rather than rebuild.

In [ ]:
# Cell 6 — save figure

fig_path = figures / "23_surviving_residues.png"

ax = df.plot.bar(
    x="residue",
    y="retention_score",
    legend=False,
    figsize=(10, 5),
)

ax.set_title("Surviving residues after pruning")
ax.set_xlabel("Residue")
ax.set_ylabel("Retention score")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(fig_path, dpi=160)
plt.show()

print("saved:", fig_path)
print("exists:", fig_path.exists())

In [ ]:
# Cell 7 — verify outputs

expected = [
    results / "23_surviving_residues.csv",
    figures / "23_surviving_residues.png",
]

for path in expected:
    print(path, "exists:", path.exists())

## Result

Notebook 23 creates:

```text
results/23_surviving_residues.csv
figures/23_surviving_residues.png
```

Next notebook:

```text
notebooks/29_budget_matched_comparisons.ipynb
```

Next question:

> If compute budgets are matched, when does revision still outperform reset?